In [4]:
import os
import pandas as pd
import kagglehub

# 1. Download the latest version of the Chicago Crime Dataset
path = kagglehub.dataset_download("aliafzal9323/chicago-crime-dataset-2024-2026")
print("Path to dataset files:", path)

# 2. Define file paths
file_path = os.path.join(path, "chicago crimes.csv")
output_file = "cleaned_chicago_crimes.csv"
chunk_size = 1000

print(f"Reading from: {file_path}")
print(f"Saving cleaned data to: {output_file}")

# 3. Process and save the data chunk-by-chunk
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):

    # --- CLEANING & TRANSFORMATION STEPS ---

    # Handle missing values for categorical column
    chunk['Location Description'] = chunk['Location Description'].fillna("Unknown")

    # Drop rows where critical geospatial coordinates are missing
    chunk = chunk.dropna(
        subset=[
            'X Coordinate',
            'Y Coordinate',
            'Latitude',
            'Longitude',
            'Location'
        ]
    )

    # General cleanup (Drop any remaining NaN records)
    chunk = chunk.dropna()

    # Drop duplicated entries found within this chunk
    chunk = chunk.drop_duplicates()

    # Fix date time strings to proper datetime format
    chunk["Updated On"] = pd.to_datetime(
        chunk["Updated On"],
        format="%m/%d/%Y %I:%M:%S %p",
        errors="coerce"
    )

    # Optimise data types to reduce memory footprints
    chunk["Latitude"] = chunk["Latitude"].astype("float32")
    chunk["Longitude"] = chunk["Longitude"].astype("float32")
    chunk["Beat"] = chunk["Beat"].astype("int32")

    # --- SEQUENTIAL SAVING STEP ---

    if i == 0:
        # Write the very first chunk and include the column headers ('w' = write mode)
        chunk.to_csv(output_file, mode="w", index=False, header=True)
        print("First chunk processed and file initialized with headers.")
    else:
        # Append all subsequent chunks without duplicating headers ('a' = append mode)
        chunk.to_csv(output_file, mode="a", index=False, header=False)

print("\nProcessing complete! The fully cleaned dataset has been saved.")

Using Colab cache for faster access to the 'chicago-crime-dataset-2024-2026' dataset.
Path to dataset files: /kaggle/input/chicago-crime-dataset-2024-2026
Reading from: /kaggle/input/chicago-crime-dataset-2024-2026/chicago crimes.csv
Saving cleaned data to: cleaned_chicago_crimes.csv
First chunk processed and file initialized with headers.

Processing complete! The fully cleaned dataset has been saved.
